# Ch06_10 — Bonus: SAC (Soft Actor-Critic) — off-policy continuous control

PPO jest **on-policy**: dane muszą pochodzić z aktualnej polityki, więc sample-efficiency bywa gorsza.

SAC jest **off-policy**:
- zbiera doświadczenia do *replay buffer*,
- uczy się z batchy próbek z przeszłości,
- dodaje regularizację entropii (soft), co stabilizuje eksplorację.

## Kluczowe elementy SAC

- Aktor: $\pi_\theta(a|s)$ (Gaussian + tanh)
- Dwa krytyki: $Q_{\phi_1}(s,a)$, $Q_{\phi_2}(s,a)$ (redukcja overestimation)
- Target networks: $\phi'_1,\phi'_2$ (soft update $\tau$)
- Entropy temperature $\alpha$ (tu: stała dla prostoty)

### Update (intuicyjnie)

1) Target dla Q:

$
y = r + \gamma(1-d)\left(\min(Q'_1,Q'_2)(s',a') - \alpha \log\pi(a'|s')\right)
$

2) Krytyki minimalizują MSE do $y$.

3) Aktor maksymalizuje:

$
\mathbb{E}\left[\min(Q_1,Q_2)(s,a) - \alpha \log\pi(a|s)\right]
$

---

## Kiedy pokazać SAC?

- jako „modern off-policy baseline” (szczególnie w MuJoCo),
- jako rozszerzenie projektu domowego.

W tym notebooku dajemy minimalny kod + konfigurację.


In [1]:
# !pip install gymnasium
# !pip install "gymnasium[mujoco]"

import numpy as np
import matplotlib.pyplot as plt

from policy_based_continuous import train_sac


## Konfiguracja

SAC często potrzebuje więcej kroków niż szybkie demo PPO,
ale jest sample-efficient i stabilny.

Na zajęciach:
- ustaw `env_id="InvertedPendulum-v4"` albo `Pendulum-v1`
- `total_steps` rzędu 50k–200k

Projekt:
- Hopper/Walker2d/HalfCheetah i `total_steps` 0.5M–2M+


In [2]:
cfg = dict(
    env_id="Hopper-v4",       # na demo: "Pendulum-v1" lub "InvertedPendulum-v4"
    seed=0,
    total_steps=200_000,
    start_steps=10_000,
    update_after=1_000,
    update_every=50,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    alpha=0.2,
    hidden_sizes=(256, 256),
)

try:
    actor, (q1, q2), log = train_sac(**cfg)
except ImportError as e:
    print(e)
    print("\nUstawiam env_id='Pendulum-v1' jako fallback...\n")
    cfg["env_id"] = "Pendulum-v1"
    actor, (q1, q2), log = train_sac(**cfg)

log.steps[:3], log.avg_return[:3]


/home/piotr/anaconda3/lib/python3.8/site-packages/torch/cuda/__init__.py:128: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/home/piotr/anaconda3/lib/python3.8/site-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


KeyboardInterrupt: 

In [ ]:
plt.figure()
plt.plot(log.steps, log.avg_return)
plt.xlabel("environment steps")
plt.ylabel("avg episodic return (window=10 ep)")
plt.title(f"SAC on {cfg['env_id']}")
plt.show()


## Uwagi

Ta wersja SAC jest celowo uproszczona (pod dydaktykę):
- $\alpha$ jest stałe (pełne SAC często uczy $\alpha$ automatycznie),
- brak normalizacji obserwacji/reward,
- brak „policy delay” itp.

Jeśli będziecie chcieli to rozbudować jako projekt:
- automatyczne \(\alpha\) (target entropy),
- normalizacja obserwacji (RunningMeanStd),
- większe batch size / tuning LR.
